# Mobilitätsauslastungsprognose: Veranstaltungsdaten

## 1. Installation wichtiger Pakete

In [ ]:
# pip install requests
# pip install holidays

## 2. Bibliotheken importieren

In [ ]:
# === Standardbibliotheken ===
import json
import base64
import time
import re
import unicodedata
from urllib.parse import urlparse, parse_qs
from datetime import datetime, timedelta, time

# === Datenverarbeitung ===
import pandas as pd
import numpy as np

# === HTTP-Anfragen & Dateioperationen ===
import requests
import fsspec

# === Maschinelles Lernen ===
from sklearn.neighbors import BallTree

# === Spark ===
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit
from pyspark.sql.types import IntegerType

# === Feiertage ===
import holidays

## 3. Spark Session erstellen

In [ ]:
# Falls noch keine Spark-Session existiert, wird eine neue erstellt
spark = SparkSession.builder.appName("Import Weather Data").getOrCreate()

## 4. Einlesen der Daten aus Veranstaltungsdatenbank 

In [2]:
# -----------------------------
# 0. Konfiguration & Authentifizierung
# -----------------------------
# ADLS Gen2 Pfad
output_path_base = (
    'abfss://processeddata@stdevdpenricheddl001.dfs.core.windows.net/'
    'Mobilität_Auslastungsprognose/Auslastung/json_files/'
)

# HTTP-Basic Auth für API
session = requests.Session()
session.trust_env = False
creds = 'csawadogo:2a258RTR§>hg'.encode('utf-8')
token = base64.b64encode(creds).decode('ascii')
headers = {
    'Authorization': f'Basic {token}',
    'Accept': 'application/json'
}

# fsspec für ADLS Gen2
fs = fsspec.filesystem(
    'abfs',
    account_name='stdevdpenricheddl001'
    # ggf. weitere Auth-Parameter (SAS, OAuth) hier
)

In [3]:
# -----------------------------
# 1. Zähl-Funktion für Events bis Ende 2022
# -----------------------------
def count_events_before_2023():
    """
    Ermittelt die Gesamtzahl der Events mit starts_before 2023-01-01 über das letzte Seiten-URL-Feld.
    """
    url = 'https://www.datenportal-muensterland.de/api/v1/events'
    params = {
        'page[size]': 1,
        'page[number]': 1,
        'filter[starts_before]': '2023-01-01T00:00:00'
    }
    resp = session.get(url, headers=headers, params=params)
    resp.raise_for_status()
    data = resp.json()
    last_page_url = data.get('last_page_url') or data.get('links', {}).get('last')
    if last_page_url:
        parsed = urlparse(last_page_url)
        page_num = int(parse_qs(parsed.query)['page[number]'][0])
        print(f"Anzahl Events bis Ende 2022: {page_num}")
        return page_num
    else:
        print('last_page_url nicht gefunden; kein Paging verfügbar.')
        return 0


In [4]:
# -----------------------------
# 2. Pagination-Funktion (page[size]/page[number])
# -----------------------------
def fetch_all(endpoint, page_size=200, filters=None):
    """
    Ruft alle Datensätze eines Endpunkts paginiert ab (bis page_size max 200).
    filters als dict für filter[...] oder sort, append.
    """
    base = f'https://www.datenportal-muensterland.de/api/v1/{endpoint}'
    page = 1
    all_data = []
    while True:
        params = {
            'page[size]': page_size,
            'page[number]': page
        }
        if filters:
            params.update(filters)
        r = session.get(base, headers=headers, params=params)
        r.raise_for_status()
        payload = r.json()
        page_data = payload.get('data', [])
        if not page_data:
            break
        all_data.extend(page_data)
        print(f"{endpoint}: Seite {page} geladen ({len(page_data)} Einträge)")
        page += 1
    print(f"{endpoint}: Gesamt {len(all_data)} Einträge geladen.")
    return all_data

In [5]:
# -----------------------------
# -----------------------------
# 3. Daten abrufen und in ADLS speichern
# -----------------------------
# Datum für dynamisches Ende (bis heute)

end_date = datetime.datetime.now().strftime('%Y-%m-%dT%H:%M:%S')

endpoints = ['events', 'public_transport_stops', 'pois']
filters = {
    'events': {
        'filter[starts_after]': '2022-01-01T00:00:00',
        'filter[starts_before]': end_date
    }
}

start_total = time.perf_counter()
# Zähle erst
total_events = count_events_before_2023()

for ep in endpoints:
    t0 = time.perf_counter()
    data = fetch_all(ep, page_size=200, filters=filters.get(ep))
    obj = {'data': data}
    path = output_path_base + f'{ep}.json'
    with fs.open(path, 'w') as f:
        json.dump(obj, f, ensure_ascii=False, indent=4)
    print(f"{ep} gespeichert: {len(data)} Einträge in {time.perf_counter()-t0:.2f}s -> {path}")
print(f"Gesamtdauer Abruf+Speichern: {time.perf_counter()-start_total:.2f}s")


In [6]:
# -----------------------------
# 4. Einlesen in pandas
# -----------------------------
with fs.open(output_path_base + 'events.json','r') as f:
    events_json = json.load(f)
with fs.open(output_path_base + 'public_transport_stops.json','r') as f:
    stops_json = json.load(f)
with fs.open(output_path_base + 'pois.json','r') as f:
    pois_json = json.load(f)

df_events = pd.json_normalize(events_json['data'])
df_stops = pd.json_normalize(stops_json['data'])
df_pois   = pd.json_normalize(pois_json['data'])
print(f"DataFrames geladen: events={df_events.shape}, stops={df_stops.shape}, pois={df_pois.shape}")

In [7]:
# -----------------------------
# 5. Spaltenreduzierung für Merge
# -----------------------------
# Events: id, start, end, poi.id, poi coords
df_events = df_events[['id', 'name', 'start_datetime','end_datetime','poi.id','poi.address.latitude','poi.address.longitude']]
# Haltestellen: id,name, coords
df_stops = df_stops[['id','name','address.latitude','address.longitude']]
# POIs: poi.id, coords, opening_hours
df_pois = df_pois[['id','name', 'address.latitude','address.longitude','opening_hours']]
df_pois.rename(columns={'id':'poi.id'}, inplace=True)
print(f"reduced: events={df_events.shape}, stops={df_stops.shape}, pois={df_pois.shape}")


In [9]:
# Step 1: Filter rows where name contains "Münster "
df_stops = df_stops[df_stops["name"].str.contains("Münster ", na=False)].copy()

# Step 2: Remove "Münster " at the start, unless it starts with "Münster (Westf"
def clean_name(name):
    if name.startswith("Münster ") and not name.startswith("Münster (Westf"):
        return name.replace("Münster ", "", 1)  # Only the first occurrence
    return name

df_stops["name"] = df_stops["name"].apply(clean_name)

# Step 3: Count remaining rows
print(f"Remaining rows after cleaning: {len(df_stops)}")


In [10]:
# Nur Events in Münster (z. B. grob eingekreist)
df_events = df_events[
    (df_events["poi.address.latitude"] >= 51.85) &
    (df_events["poi.address.latitude"] <= 51.95) &
    (df_events["poi.address.longitude"] >= 7.5) &
    (df_events["poi.address.longitude"] <= 7.7)
].reset_index(drop=True)

df_events.head()

In [12]:
# Explizit in datetime (inkl. Zeitzonen-Unterstützung)
df_events["start_datetime"] = pd.to_datetime(df_events["start_datetime"], utc=True)
df_events["end_datetime"] = pd.to_datetime(df_events["end_datetime"], utc=True)

# Optional: Zeitzone entfernen (wenn nicht gebraucht)
df_events["start_datetime"] = df_events["start_datetime"].dt.tz_convert(None)
df_events["end_datetime"] = df_events["end_datetime"].dt.tz_convert(None)


# Extrahiere das Datum (nur aus dem Startzeitpunkt)
df_events["start_date"] = df_events["start_datetime"].dt.date
df_events["end_date"] = df_events["end_datetime"].dt.date

# Extrahiere Start- und Endstunde
df_events["start_hour"] = df_events["start_datetime"].dt.hour
df_events["end_hour"] = df_events["end_datetime"].dt.hour

df_events = df_events.drop(columns=["start_datetime", "end_datetime"])

df_events = df_events.sort_values(by=["start_date", "start_hour"]).reset_index(drop=True)

display(df_events.tail(200))

## 5. Anreichern von weiteren Veranstaltungen: Weihnachtsmärkte

In [13]:
# Weihnachtsmärkte 2023 + 2024 (vereinigt, falls gewünscht getrennt, einfach aufteilen)
weihnachtsmaerkte = [
    {
        "name": "Giebelhüüskesmarkt",
        "lat": 51.964020842268326,
        "lon": 7.6231940128560485,
        "start": "2023-11-27",
        "end": "2023-12-23"
    },
    {
        "name": "Weihnachtsmarkt am Kiepenkerl",
        "lat": 51.96435182268396,
        "lon": 7.626373135164962,
        "start": "2023-11-27",
        "end": "2023-12-23"
    },
    {
        "name": "Lichtermarkt St. Lamberti",
        "lat": 51.96659878845625,
        "lon": 7.6296453275109055,
        "start": "2023-11-27",
        "end": "2023-12-22"
    },
    {
        "name": "Weihnachtsmarkt ums Rathaus",
        "lat": 51.96495942433944,
        "lon": 7.630326672931803,
        "start": "2023-11-27",
        "end": "2023-12-22"
    },
    {
        "name": "Aegidii-Weihnachtsmarkt",
        "lat": 51.96462231705278,
        "lon": 7.625332318368143,
        "start": "2023-11-27",
        "end": "2023-12-23"
    },
    {
        "name": "X-MS Weihnachtsmarkt am Harsewinkelplatz",
        "lat": 51.96335009733814,
        "lon": 7.631002595525664,
        "start": "2023-11-27",
        "end": "2023-12-23"
    },
]

# Öffnungszeiten
def get_opening_hours(weekday):
    # 0 = Montag, ..., 6 = Sonntag
    if weekday in [4, 5]:  # Freitag, Samstag
        return 11, 21
    else:  # Sonntag - Donnerstag
        return 11, 20

# Liste für neue Event-Zeilen
new_events = []

for markt in weihnachtsmaerkte:
    start_date = datetime.strptime(markt["start"], "%Y-%m-%d")
    end_date = datetime.strptime(markt["end"], "%Y-%m-%d")
    delta = (end_date - start_date).days + 1

    for i in range(delta):
        day = start_date + timedelta(days=i)
        start_hour, end_hour = get_opening_hours(day.weekday())

        new_events.append({
            "name": markt["name"],
            "start_date": day.date(),
            "end_date": day.date(),  # Tages-Events
            "start_hour": start_hour,
            "end_hour": end_hour,
            "poi.address.latitude": markt["lat"],
            "poi.address.longitude": markt["lon"]
        })

# In DataFrame umwandeln
weihnachts_df = pd.DataFrame(new_events)

# Falls df_events die gleichen Spaltenstruktur hat, einfach anhängen:
df_events = pd.concat([df_events, weihnachts_df], ignore_index=True)

# Sortieren
df_events = df_events.sort_values(by=["start_date", "start_hour"]).reset_index(drop=True)

# Ausgabe prüfen
display(df_events.tail(20))


## 6. Anreichern von weiteren Veranstaltungen: Send

In [15]:
# Koordinaten für alle Send-Events
send_lat = 51.96354180842706
send_lon = 7.614458730634429

# Alle Send-Termine (Datum, Uhrzeit, Titel)
send_events_raw = [
    # Frühjahrssend 2023
    ("Frühjahrssend 2023", "2023-03-11", 14, 24),
    ("Frühjahrssend 2023", "2023-03-12", 11, 23),
    ("Frühjahrssend 2023", "2023-03-13", 14, 23),
    ("Frühjahrssend 2023", "2023-03-14", 14, 23),
    ("Frühjahrssend 2023", "2023-03-15", 14, 23),
    ("Frühjahrssend 2023", "2023-03-16", 14, 23),
    ("Frühjahrssend 2023", "2023-03-17", 14, 24),
    ("Frühjahrssend 2023", "2023-03-18", 14, 24),
    ("Frühjahrssend 2023", "2023-03-19", 11, 22),

    # Sommersend 2023
    ("Sommersend 2023", "2023-07-13", 14, 23),
    ("Sommersend 2023", "2023-07-14", 14, 24),
    ("Sommersend 2023", "2023-07-15", 14, 24),
    ("Sommersend 2023", "2023-07-16", 11, 23),
    ("Sommersend 2023", "2023-07-17", 14, 23),

    # Herbstsend 2023
    ("Herbstsend 2023", "2023-10-21", 14, 23),
    ("Herbstsend 2023", "2023-10-22", 11, 23),
    ("Herbstsend 2023", "2023-10-23", 14, 23),
    ("Herbstsend 2023", "2023-10-24", 14, 23),
    ("Herbstsend 2023", "2023-10-25", 14, 23),
    ("Herbstsend 2023", "2023-10-27", 14, 23),
    ("Herbstsend 2023", "2023-10-28", 14, 24),
    ("Herbstsend 2023", "2023-10-29", 11, 22),

    # Frühjahressend 2024
    ("Frühjahrssend 2024", "2024-03-02", 14, 24),
    ("Frühjahrssend 2024", "2024-03-03", 11, 23),
    ("Frühjahrssend 2024", "2024-03-04", 14, 23),
    ("Frühjahrssend 2024", "2024-03-05", 14, 23),
    ("Frühjahrssend 2024", "2024-03-06", 14, 23),
    ("Frühjahrssend 2024", "2024-03-07", 14, 23),
    ("Frühjahrssend 2024", "2024-03-08", 14, 24),
    ("Frühjahrssend 2024", "2024-03-09", 14, 24),
    ("Frühjahrssend 2024", "2024-03-10", 11, 22),

    # Sommersend 2024
    ("Sommersend 2024", "2024-07-18", 14, 23),
    ("Sommersend 2024", "2024-07-19", 14, 24),
    ("Sommersend 2024", "2024-07-20", 14, 24),
    ("Sommersend 2024", "2024-07-21", 11, 23),
    ("Sommersend 2024", "2024-07-22", 14, 23),

    # Herbstsend 2024
    ("Herbstsend 2024", "2024-10-26", 14, 24),
    ("Herbstsend 2024", "2024-10-27", 11, 23),
    ("Herbstsend 2024", "2024-10-28", 14, 23),
    ("Herbstsend 2024", "2024-10-29", 14, 23),
    ("Herbstsend 2024", "2024-10-30", 14, 23),
    ("Herbstsend 2024", "2024-10-31", 14, 23),
    ("Herbstsend 2024", "2024-11-01", 14, 24),
    ("Herbstsend 2024", "2024-11-02", 14, 24),
    ("Herbstsend 2024", "2024-11-03", 11, 22),

    # Frühjahressend 2025
    ("Frühjahrssend 2025", "2025-03-22", 14, 24),
    ("Frühjahrssend 2025", "2025-03-23", 11, 23),
    ("Frühjahrssend 2025", "2025-03-24", 14, 23),
    ("Frühjahrssend 2025", "2025-03-25", 14, 23),
    ("Frühjahrssend 2025", "2025-03-26", 14, 23),
    ("Frühjahrssend 2025", "2025-03-27", 14, 23),
    ("Frühjahrssend 2025", "2025-03-28", 14, 24),
    ("Frühjahrssend 2025", "2025-03-29", 14, 24),
    ("Frühjahrssend 2025", "2025-03-30", 11, 22),
]

# In DataFrame konvertieren
send_df = pd.DataFrame(send_events_raw, columns=["name", "start_date", "start_hour", "end_hour"])

# Ergänzen um weitere Felder
send_df["start_date"] = pd.to_datetime(send_df["start_date"]).dt.date
send_df["end_date"] = send_df["start_date"]
send_df["poi.address.latitude"] = send_lat
send_df["poi.address.longitude"] = send_lon

# Falls du df_events schon hast:
df_events = pd.concat([df_events, send_df], ignore_index=True)
df_events = df_events.sort_values(by=["start_date", "start_hour"]).reset_index(drop=True)

# Kontrolle
display(df_events.tail(20))


## 6. Anreichern von weiteren Veranstaltungen: Preußen Münster Heimspiele

In [16]:
# 📍 Stadionkoordinaten
lat = 51.9312746983589
lon = 7.625445852726836

# 🗓️ Liste aller Spiele (Datum, Anstoßzeit)
heimspiele = [
    ("2023-02-05", "14:00"), ("2023-02-18", "14:00"), ("2023-03-04", "14:00"),
    ("2023-03-18", "14:00"), ("2023-04-08", "14:00"), ("2023-04-22", "14:00"),
    ("2023-04-28", "19:30"), ("2023-05-13", "14:00"), ("2023-08-05", "14:00"),
    ("2023-08-22", "19:00"), ("2023-09-02", "14:00"), ("2023-09-06", "19:00"),
    ("2023-09-23", "14:00"), ("2023-09-26", "20:45"), ("2023-10-04", "19:00"),
    ("2023-10-15", "16:30"), ("2023-10-24", "19:30"), ("2023-11-05", "16:30"),
    ("2023-11-18", "13:00"), ("2023-11-26", "19:30"), ("2023-12-10", "19:30"),
    ("2024-01-21", "13:30"), ("2024-01-28", "13:30"), ("2024-02-10", "14:00"),
    ("2024-02-23", "19:00"), ("2024-03-09", "16:30"), ("2024-03-30", "14:00"),
    ("2024-04-06", "14:00"), ("2024-04-21", "16:30"), ("2024-05-05", "13:30"),
    ("2024-05-18", "13:30"), ("2024-08-11", "13:30"), ("2024-08-24", "13:00"),
    ("2024-08-27", "20:45"), ("2024-09-13", "18:30"), ("2024-09-28", "20:30"),
    ("2024-10-19", "13:00"), ("2024-11-01", "18:20"), ("2024-11-22", "18:20"),
    ("2024-12-07", "20:30"), ("2024-12-21", "13:00"), ("2025-01-18", "13:00"),
    ("2025-02-07", "18:30"), ("2025-02-22", "13:00"), ("2025-03-09", "13:30"),
    ("2025-03-30", "13:30")
]

heimspiele_events = []

for datum, anstoss in heimspiele:
    # Anstoßzeit in datetime
    anstoss_dt = datetime.strptime(f"{datum} {anstoss}", "%Y-%m-%d %H:%M")
    
    # 30 Minuten vorher & 120 Minuten nachher
    start_dt = anstoss_dt - timedelta(minutes=30)
    end_dt = anstoss_dt + timedelta(minutes=120)  # 90 Spiel + 30 Nachlauf

    heimspiele_events.append({
        "name": "Preußen Münster Heimspiel",
        "start_date": start_dt.date(),
        "end_date": end_dt.date(),
        "start_hour": start_dt.hour,
        "end_hour": end_dt.hour if end_dt.date() == start_dt.date() else 23,  # falls Event über Mitternacht geht
        "poi.address.latitude": lat,
        "poi.address.longitude": lon
    })

# In DataFrame umwandeln
heimspiele_df = pd.DataFrame(heimspiele_events)

# Zu df_events hinzufügen
df_events = pd.concat([df_events, heimspiele_df], ignore_index=True)
df_events = df_events.sort_values(by=["start_date", "start_hour"]).reset_index(drop=True)

# Anzeige
display(df_events.tail(20))


## 7. Anreichern von weiteren Veranstaltungen: Weitere Veranstaltungen

In [17]:
# Rohdaten: (Name, Datum, Startzeit, Endzeit, lat, lon)
festival_events_raw = [
    # Docklands 2023
    ("Docklands Festival 2023", "2023-06-10", "12:00", "12:00", 51.944695474102325, 7.638856897887219),

    # Münster Mittendrin 2023 (alle 3 Einträge separat)
    ("Münster Mittendrin 2023", "2023-03-18", "10:00", "24:00", 51.96175135900906, 7.628002226019007),
    ("Münster Mittendrin 2023", "2023-03-18", "10:00", "24:00", 51.96175135900906, 7.628002226019007),
    ("Münster Mittendrin 2023", "2023-03-18", "11:00", "24:00", 51.96175135900906, 7.628002226019007),

    # Vainstream 2023
    ("Vainstream Rockfest 2023", "2023-06-24", "08:00", "24:00", 51.944695474102325, 7.638856897887219),

    # Docklands 2024
    ("Docklands Festival 2024", "2024-06-08", "12:00", "12:00", 51.944695474102325, 7.638856897887219),

    # Vainstream 2024 – beide Tage
    ("Vainstream Rockfest 2024", "2024-06-28", "12:00", "12:00", 51.944695474102325, 7.638856897887219),
    ("Vainstream Rockfest 2024", "2024-06-29", "12:00", "12:00", 51.944695474102325, 7.638856897887219),

    # Münster Mittendrin 2024
    ("Münster Mittendrin 2024", "2024-08-16", "12:30", "23:59", 51.96175135900906, 7.628002226019007),
    ("Münster Mittendrin 2024", "2024-08-17", "12:00", "12:00", 51.96175135900906, 7.628002226019007),
    ("Münster Mittendrin 2024", "2024-08-18", "12:00", "12:00", 51.96175135900906, 7.628002226019007),
]

# DataFrame aufbauen
festival_events = []

for name, date_str, start_str, end_str, lat, lon in festival_events_raw:
    # Umwandlung in datetime
    date_obj = datetime.strptime(date_str, "%Y-%m-%d").date()

    # Sonderfall: "24:00" → rechne auf 23:59
    if end_str == "24:00":
        end_hour = 23
    else:
        end_hour = datetime.strptime(end_str, "%H:%M").hour

    start_hour = datetime.strptime(start_str, "%H:%M").hour

    festival_events.append({
        "name": name,
        "start_date": date_obj,
        "end_date": date_obj,
        "start_hour": start_hour,
        "end_hour": end_hour,
        "poi.address.latitude": lat,
        "poi.address.longitude": lon
    })

# Umwandeln in DataFrame
festival_df = pd.DataFrame(festival_events)

# Zusammenführen mit df_events
df_events = pd.concat([df_events, festival_df], ignore_index=True)
df_events = df_events.sort_values(by=["start_date", "start_hour"]).reset_index(drop=True)

# Ausgabe prüfen
display(df_events.tail(20))

In [18]:
# -----------------------------
# 1. Stops & Events mit gültigen Koordinaten filtern
# -----------------------------

stops_clean = df_stops.dropna(subset=["address.latitude", "address.longitude"]).reset_index(drop=True)
events_clean = df_events.dropna(subset=["poi.address.latitude", "poi.address.longitude"]).reset_index()

# -----------------------------
# 2. Koordinaten vorbereiten (in Radians)
# -----------------------------

earth_radius_m = 6371000  # Erdradius in Metern

coords_stops = np.radians(stops_clean[["address.latitude", "address.longitude"]].values)

# -----------------------------
# 3. KD-Tree mit Haversine-Distanz aufbauen
# -----------------------------

tree = BallTree(coords_stops, metric="haversine")

# -----------------------------
# 4. Basisradius in Metern
# -----------------------------

default_radius_m = 500
special_radius_m = 3000  # Für "Münster Mittendrin"

# -----------------------------
# 5. Dynamische Suche & Zuweisung
# -----------------------------

df_events["nearby_stop_names"] = pd.Series([[]] * len(df_events), dtype=object)
used_radii = []

for i, event_idx in enumerate(events_clean["index"]):
    event_name = df_events.loc[event_idx, "name"]
    
    # Dynamischer Radius
    if "Münster Mittendrin" in event_name:
        radius_m = special_radius_m
    else:
        radius_m = default_radius_m

    radius_rad = radius_m / earth_radius_m
    
    # Event-Koordinaten
    event_coord = np.radians([[df_events.loc[event_idx, "poi.address.latitude"],
                               df_events.loc[event_idx, "poi.address.longitude"]]])
    
    stop_idxs = tree.query_radius(event_coord, r=radius_rad)[0]
    stop_names = stops_clean.iloc[stop_idxs]["name"].tolist()
    
    df_events.at[event_idx, "nearby_stop_names"] = stop_names
    used_radii.append(radius_m)

# Radius als neue Spalte dokumentieren
df_events.loc[events_clean["index"], "radius_used_m"] = used_radii

df_events["event_type"] = df_events["id"].apply(lambda x: 1 if pd.isna(x) else 2)

# Kontrolle
display(df_events.head())


In [19]:
df_events = df_events[["name", "start_date", "end_date", "start_hour", "end_hour", "nearby_stop_names", "event_type"]]

# Neue Liste für "aufgeklappte" Zeilen
expanded_rows = []

# Iteration über alle Events
for _, row in df_events.iterrows():
    event_name = row["name"]
    stop_names = row["nearby_stop_names"]
    start_date = row["start_date"]
    end_date = row["end_date"]
    start_hour = row["start_hour"]
    end_hour = row["end_hour"]
    event_type = row["event_type"]

    # Für jedes Datum im Intervall
    current_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    while current_date <= end_date:
        # Für jede Stunde im Intervall
        for hour in range(start_hour, end_hour + 1):
            for stop in stop_names:
                expanded_rows.append({
                    "event_name": event_name,
                    "stop_name": stop,
                    "date": current_date.date(),
                    "hour": hour,
                    "event_type": event_type
                })
        current_date += timedelta(days=1)

# In DataFrame umwandeln
df_event_expanded = pd.DataFrame(expanded_rows)

# Ausgabe prüfen
display(df_event_expanded.head(20))

In [20]:
# Zeitbereich definieren
start_date = datetime(2018, 1, 1)
end_date = datetime(2025, 4, 15)

# Alle Datumswerte im Zeitraum erzeugen
date_range = pd.date_range(start=start_date, end=end_date, freq='D')

# Alle Kombinationen von Datum und Stunde (0–23)
full_time_rows = []

for date in date_range:
    for hour in range(24):
        full_time_rows.append({
            "date": date.date(),
            "hour": hour,
            "event_name": None,
            "stop_name": None
        })

# In DataFrame umwandeln
df_empty_fill = pd.DataFrame(full_time_rows)

# Umordnen für gleiche Struktur wie df_event_expanded
df_empty_fill = df_empty_fill[["event_name", "stop_name", "date", "hour"]]

# Original erweitern
df_combined = pd.concat([df_event_expanded, df_empty_fill], ignore_index=True)

# Optional: sortieren
df_combined = df_combined.sort_values(by=["date", "hour"]).reset_index(drop=True)

# Kontrolle
display(df_combined.head(20))


## 8. Hinzufügen von einer zeitlichen Einordnung bezüglich COVID-19 

In [21]:
# Zeitgrenzen definieren
covid_start = datetime(2020, 3, 11)
covid_end = datetime(2023, 1, 31)

# covid_period-Spalte zuweisen
df_combined["covid_period"] = pd.to_datetime(df_combined["date"]).apply(
    lambda d: "Pre-COVID" if d < covid_start
    else "During COVID" if d <= covid_end
    else "Post-COVID"
)

## 9. Hinzufügen Feiertagen /Schulferien

In [22]:
# Spark-Session starten
spark = SparkSession.builder.getOrCreate()

# --- 1️⃣ Bestehendes pandas-DF in Spark übernehmen ---
df_combined_spark = spark.createDataFrame(df_combined)

# --- 2️⃣ Gesetzliche Feiertage NRW als Spark-DF ---
nrw_holidays = holidays.Germany(years=range(2022, 2026), subdiv='NW')
df_pub_pd = pd.DataFrame([{"date": pd.to_datetime(d)} for d in nrw_holidays.keys()])
df_pub_spark = (
    spark.createDataFrame(df_pub_pd)
         .withColumn("is_public_holiday", lit(1))
)

# --- 3️⃣ Schulferien NRW von der API holen ---
response = requests.get("https://ferien-api.de/api/v1/holidays?state=NW")
df_ferien = pd.DataFrame(response.json())
df_ferien["start"] = pd.to_datetime(df_ferien["start"])
df_ferien["end"]   = pd.to_datetime(df_ferien["end"])
df_ferien_spark = spark.createDataFrame(df_ferien[["start", "end"]])

# --- 4️⃣ Joins und eigene Spalten erzeugen ---
df_result = (
    df_combined_spark
      # 4.1 Join auf öffentliche Feiertage
      .join(df_pub_spark, on="date", how="left")
      # 4.2 Join auf Schulferien-Intervalle
      .join(
          df_ferien_spark,
          (col("date") >= col("start")) & (col("date") <= col("end")),
          how="left"
      )
      # 4.3 Neue Spalte is_school_holiday
      .withColumn(
          "is_school_holiday",
          when(col("start").isNotNull(), lit(1)).otherwise(lit(0))
      )
      # 4.4 Nulls in is_public_holiday zu 0 casten
      .withColumn(
          "is_public_holiday",
          when(col("is_public_holiday") == 1, lit(1)).otherwise(lit(0))
      )
      # 4.5 Hilfsspalten entfernen
      .drop("start", "end")
)

# --- 5️⃣ Ergebnis prüfen ---
df_result.orderBy("date").show(20)


In [23]:
final_df = df_result.fillna({"event_type": 0}) \
    .withColumn("event_type", col("event_type").cast(IntegerType()))

final_df = final_df.drop("event_name")
final_df.show(n=20)

In [24]:
# Step 8: Write the resulting merged DataFrame to a Parquet file in the Azure Data Lake storage
path = 'data/event_data'
final_df.write.mode("overwrite").parquet(path)